In [1]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELDA 1 — Librerías, Estilo Global y Carga de CSVs     ║
# ╚══════════════════════════════════════════════════════════╝

import os
import glob
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.figure import Figure
import ipywidgets as widgets
from IPython.display import display, clear_output
from io import BytesIO
import base64
import chardet

# ── Estilo global Matplotlib ────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'       : 'DejaVu Sans',
    'font.size'         : 10,
    'axes.titlesize'    : 12,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 10,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'grid.alpha'        : 0.3,
    'grid.linestyle'    : '--',
    'figure.dpi'        : 110,
})

PALETA = ['#2563EB','#0D9488','#EA580C','#7C3AED','#16A34A','#DC2626','#D97706','#0891B2']

# ── Directorio de datos ─────────────────────────────────────────────────────
CSV_DIR = 'csv_dashboard'

def cargar_todos_los_csv(directorio: str) -> dict[str, pd.DataFrame]:
    """Lee todos los .csv del directorio y devuelve {nombre_sin_ext: DataFrame}."""
    catalogos = {}
    rutas = sorted(glob.glob(os.path.join(directorio, '*.csv')))
    if not rutas:
        print(f"⚠️  No se encontraron archivos CSV en '{directorio}/'")
        return catalogos
    for ruta in rutas:
        nombre = os.path.splitext(os.path.basename(ruta))[0]
        df = None
        
        # Detectar encoding con chardet
        with open(ruta, 'rb') as f:
            result = chardet.detect(f.read())
            detected_enc = result.get('encoding', 'utf-8')
        
        # Intentar con encoding detectado y fallbacks
        encodings = [detected_enc, 'utf-8', 'latin-1', 'utf-16', 'iso-8859-1', 'cp1252']
        separators = [',', ';', '\t', '|']
        
        for enc in encodings:
            if enc is None:
                continue
            for sep in separators:
                try:
                    df = pd.read_csv(ruta, encoding=enc, sep=sep, on_bad_lines='skip')
                    if len(df.columns) > 1:  # Si tiene múltiples columnas, usar este
                        break
                    df = None  # Reset si solo tiene 1 columna
                except (UnicodeDecodeError, pd.errors.ParserError, Exception):
                    continue
            if df is not None and len(df.columns) > 1:
                break
        
        if df is None:
            print(f"  ✗  {nombre}  →  No se pudo leer (encoding inválido)")
            continue
        
        # limpiar nombres de columnas
        df.columns = df.columns.str.strip()
        catalogos[nombre] = df
        print(f"  ✓  {nombre}  →  {df.shape[0]} filas × {df.shape[1]} cols")
    return catalogos

print("=" * 62)
print("   DASHBOARD — Acceso y Rendimiento Universitario en Chile")
print("=" * 62)
print(f"\nEscaneando carpeta '{CSV_DIR}/' …\n")
DATOS = cargar_todos_los_csv(CSV_DIR)
print(f"\nTotal archivos cargados: {len(DATOS)}")
print("Celda 1 lista ✓")

   DASHBOARD — Acceso y Rendimiento Universitario en Chile

Escaneando carpeta 'csv_dashboard/' …

  ✓  ArchivoB_Adm2019  →  302973 filas × 43 cols
  ✓  todas_las_ingenierias_chile  →  306 filas × 9 cols

Total archivos cargados: 2
Celda 1 lista ✓


In [2]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELDA 2 — Motor Inteligente de Gráficos                ║
# ╚══════════════════════════════════════════════════════════╝

def _fig_to_widget(fig: Figure) -> widgets.Image:
    """Convierte una Figure de Matplotlib en un widget Image para ipywidgets."""
    buf = BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=130)
    buf.seek(0)
    plt.close(fig)
    return widgets.Image(value=buf.read(), format='png',
                         layout=widgets.Layout(width='100%', max_height='370px'))


def _clasificar_columnas(df: pd.DataFrame):
    """Clasifica columnas en numéricas, categóricas y de fecha."""
    numericas    = df.select_dtypes(include='number').columns.tolist()
    categoricas  = df.select_dtypes(include=['object','category']).columns.tolist()
    # intentar detectar fechas
    fechas = []
    for col in categoricas:
        try:
            pd.to_datetime(df[col], errors='raise')
            fechas.append(col)
        except Exception:
            pass
    categoricas = [c for c in categoricas if c not in fechas]
    return numericas, categoricas, fechas


def grafico_automatico(df: pd.DataFrame,
                       eje_x: str,
                       eje_y: str,
                       tipo: str,
                       color_por: str = None) -> widgets.Image:
    """
    Genera el gráfico más adecuado según los parámetros.
    tipo ∈ {'barras', 'linea', 'histograma', 'dispersion', 'caja', 'auto'}
    """
    numericas, categoricas, fechas = _clasificar_columnas(df)

    # ── Modo AUTO: detectar el mejor tipo ──────────────────────────────────
    if tipo == 'auto':
        if eje_x in fechas or eje_x in categoricas:
            if eje_y in numericas:
                n_cats = df[eje_x].nunique()
                tipo = 'linea' if (eje_x in fechas or n_cats > 12) else 'barras'
            else:
                tipo = 'barras'
        elif eje_x in numericas and eje_y in numericas:
            tipo = 'dispersion'
        elif eje_x in numericas and eje_y == '—':
            tipo = 'histograma'
        else:
            tipo = 'barras'

    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    colores = PALETA

    # ── BARRAS ─────────────────────────────────────────────────────────────
    if tipo == 'barras':
        if eje_y == '—':
            conteo = df[eje_x].value_counts().head(20)
            bars = ax.bar(conteo.index.astype(str), conteo.values,
                          color=colores[:len(conteo)], edgecolor='white', linewidth=1.2, zorder=3)
            ax.set_ylabel('Frecuencia')
            for b in bars:
                ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
                        f'{int(b.get_height()):,}', ha='center', va='bottom', fontsize=8.5)
        else:
            if color_por and color_por != '—' and color_por in df.columns:
                grupos = df.groupby([eje_x, color_por])[eje_y].mean().unstack()
                grupos.head(15).plot(kind='bar', ax=ax, color=colores[:len(grupos.columns)],
                                     edgecolor='white', linewidth=0.8, zorder=3)
                ax.set_ylabel(f'Promedio de {eje_y}')
            else:
                agrupado = df.groupby(eje_x)[eje_y].mean().head(20).sort_values(ascending=False)
                bars = ax.bar(agrupado.index.astype(str), agrupado.values,
                              color=colores[:len(agrupado)], edgecolor='white', linewidth=1.2, zorder=3)
                ax.set_ylabel(f'Promedio de {eje_y}')
                for b in bars:
                    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
                            f'{b.get_height():.1f}', ha='center', va='bottom', fontsize=8.5)
        ax.set_xlabel(eje_x)
        plt.xticks(rotation=35, ha='right')

    # ── LÍNEA ──────────────────────────────────────────────────────────────
    elif tipo == 'linea':
        if color_por and color_por != '—' and color_por in df.columns:
            for i, (grp, sub) in enumerate(df.groupby(color_por)):
                sub_ord = sub.sort_values(eje_x)
                ax.plot(sub_ord[eje_x].astype(str), sub_ord[eje_y],
                        marker='o', markersize=5, linewidth=2,
                        color=colores[i % len(colores)], label=str(grp))
            ax.legend(title=color_por, fontsize=8.5)
        else:
            agrupado = df.groupby(eje_x)[eje_y].mean().reset_index().sort_values(eje_x)
            ax.plot(agrupado[eje_x].astype(str), agrupado[eje_y],
                    marker='o', markersize=6, linewidth=2.2, color=colores[0])
            ax.fill_between(range(len(agrupado)), agrupado[eje_y].values,
                            alpha=0.12, color=colores[0])
        ax.set_xlabel(eje_x)
        ax.set_ylabel(f'Promedio de {eje_y}')
        plt.xticks(rotation=35, ha='right')

    # ── HISTOGRAMA ─────────────────────────────────────────────────────────
    elif tipo == 'histograma':
        col = eje_x if eje_y == '—' else eje_y
        if color_por and color_por != '—' and color_por in df.columns:
            for i, (grp, sub) in enumerate(df.groupby(color_por)):
                sub[col].dropna().hist(ax=ax, bins=30, alpha=0.6,
                                       color=colores[i % len(colores)],
                                       edgecolor='white', label=str(grp), density=True)
            ax.legend(title=color_por, fontsize=8.5)
        else:
            ax.hist(df[col].dropna(), bins=30, color=colores[0],
                    edgecolor='white', linewidth=0.8, zorder=3)
        ax.set_xlabel(col)
        ax.set_ylabel('Frecuencia')
        media = df[col].dropna().mean()
        ax.axvline(media, color='red', linestyle='--', linewidth=1.4,
                   label=f'Media = {media:.1f}')
        ax.legend(fontsize=8.5)

    # ── DISPERSIÓN ─────────────────────────────────────────────────────────
    elif tipo == 'dispersion':
        muestra = df[[eje_x, eje_y] + ([color_por] if color_por and color_por != '—' and color_por in df.columns else [])].dropna()
        if len(muestra) > 3000:
            muestra = muestra.sample(3000, random_state=42)
        if color_por and color_por != '—' and color_por in df.columns:
            for i, (grp, sub) in enumerate(muestra.groupby(color_por)):
                ax.scatter(sub[eje_x], sub[eje_y], alpha=0.5, s=18,
                           color=colores[i % len(colores)], label=str(grp))
            ax.legend(title=color_por, fontsize=8.5)
        else:
            ax.scatter(muestra[eje_x], muestra[eje_y], alpha=0.45, s=18,
                       color=colores[0])
        # línea de tendencia
        try:
            m, b_val = np.polyfit(muestra[eje_x], muestra[eje_y], 1)
            xs = np.linspace(muestra[eje_x].min(), muestra[eje_x].max(), 200)
            ax.plot(xs, m*xs + b_val, color='red', linewidth=1.5,
                    linestyle='--', label='Tendencia')
            ax.legend(fontsize=8.5)
        except Exception:
            pass
        ax.set_xlabel(eje_x)
        ax.set_ylabel(eje_y)

    # ── CAJA (BOXPLOT) ─────────────────────────────────────────────────────
    elif tipo == 'caja':
        grupos = [g[eje_y].dropna().values
                  for _, g in df.groupby(eje_x)]
        labels = [str(k) for k in df[eje_x].unique()]
        bp = ax.boxplot(grupos, labels=labels, patch_artist=True,
                        medianprops=dict(color='white', linewidth=2))
        for i, patch in enumerate(bp['boxes']):
            patch.set_facecolor(colores[i % len(colores)])
        ax.set_xlabel(eje_x)
        ax.set_ylabel(eje_y)
        plt.xticks(rotation=35, ha='right')

    ax.set_title(f'{eje_y}  por  {eje_x}' if eje_y != '—' else f'Distribución de {eje_x}')
    ax.set_axisbelow(True)
    fig.tight_layout()
    return _fig_to_widget(fig)


def resumen_estadistico_html(df: pd.DataFrame) -> str:
    """Genera un bloque HTML con estadísticas básicas del DataFrame."""
    numericas = df.select_dtypes(include='number').columns.tolist()
    if not numericas:
        return "<i>Sin columnas numéricas disponibles.</i>"

    filas = ""
    for col in numericas[:8]:
        s = df[col].dropna()
        filas += (f"<tr><td><b>{col}</b></td>"
                  f"<td>{s.count():,}</td>"
                  f"<td>{s.mean():.2f}</td>"
                  f"<td>{s.median():.2f}</td>"
                  f"<td>{s.std():.2f}</td>"
                  f"<td>{s.min():.1f}</td>"
                  f"<td>{s.max():.1f}</td></tr>")

    return f"""
    <div style='font-family:sans-serif; font-size:0.85em; overflow-x:auto;'>
      <table style='border-collapse:collapse; width:100%;'>
        <thead style='background:#2563EB; color:white;'>
          <tr>
            <th style='padding:6px 10px; text-align:left;'>Columna</th>
            <th style='padding:6px 8px;'>N</th>
            <th style='padding:6px 8px;'>Media</th>
            <th style='padding:6px 8px;'>Mediana</th>
            <th style='padding:6px 8px;'>DE</th>
            <th style='padding:6px 8px;'>Mín</th>
            <th style='padding:6px 8px;'>Máx</th>
          </tr>
        </thead>
        <tbody>{''.join([f'<tr style="background:{"#f8fafc" if i%2==0 else "white"};">' + filas.split('<tr>')[i+1] for i in range(len(numericas[:8]))])}</tbody>
      </table>
    </div>"""

print("Celda 2 lista ✓  —  Motor de gráficos y estadísticas cargado.")

Celda 2 lista ✓  —  Motor de gráficos y estadísticas cargado.


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELDA 3 — Interfaz Interactiva del Dashboard            ║
# ╚══════════════════════════════════════════════════════════╝

# ── Panel de selección de archivo ───────────────────────────────────────────
selector_archivo = widgets.Dropdown(
    options={nombre: nombre for nombre in sorted(DATOS.keys())},
    description='Archivo:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='280px')
)

# ── Panel de selección de eje X ─────────────────────────────────────────────
selector_x = widgets.Dropdown(
    options=['Seleccionar...'],
    description='Eje X:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='280px')
)

# ── Panel de selección de eje Y ─────────────────────────────────────────────
selector_y = widgets.Dropdown(
    options=['Seleccionar...'],
    description='Eje Y:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='280px')
)

# ── Panel de selección de tipo de gráfico ───────────────────────────────────
selector_tipo = widgets.Dropdown(
    options=['auto', 'barras', 'linea', 'histograma', 'dispersion', 'caja'],
    value='auto',
    description='Tipo:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='280px')
)

# ── Panel de selección de color ─────────────────────────────────────────────
selector_color = widgets.Dropdown(
    options=['—'],
    value='—',
    description='Color por:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='280px')
)

# ── Botón para generar gráfico ──────────────────────────────────────────────
btn_generar = widgets.Button(
    description='📊 Generar Gráfico',
    button_style='info',
    tooltip='Genera el gráfico con los parámetros seleccionados',
    layout=widgets.Layout(width='136px', height='40px')
)

# ── Botón para actualizar estadísticas ──────────────────────────────────────
btn_estadisticas = widgets.Button(
    description='📈 Estadísticas',
    button_style='success',
    tooltip='Muestra resumen estadístico del archivo',
    layout=widgets.Layout(width='136px', height='40px')
)

# ── Botón para recargar archivos ────────────────────────────────────────────
btn_recargar = widgets.Button(
    description='🔄 Recargar',
    button_style='warning',
    tooltip='Recarga los archivos CSV de la carpeta',
    layout=widgets.Layout(width='136px', height='40px')
)

# ── Área de salida para gráficos ────────────────────────────────────────────
output_grafico = widgets.Output()

# ── Área de salida para estadísticas ────────────────────────────────────────
output_estadisticas = widgets.Output()

# ── Mensaje de estado ───────────────────────────────────────────────────────
label_estado = widgets.HTML(value="<b style='color:#666;'>⏳ Listo para generar gráficos</b>")


# ── Función para recargar archivos CSV ──────────────────────────────────────
def recargar_archivos(button):
    """Recarga dinámicamente los CSV de la carpeta."""
    global DATOS
    with output_grafico:
        output_grafico.clear_output(wait=True)
        label_estado.value = "<b style='color:#EA580C;'>⏳ Recargando archivos...</b>"
        
        try:
            DATOS.clear()
            DATOS = cargar_todos_los_csv(CSV_DIR)
            
            # Actualizar opciones del selector
            nuevos_archivos = sorted(DATOS.keys())
            selector_archivo.options = {n: n for n in nuevos_archivos}
            
            if nuevos_archivos:
                selector_archivo.value = nuevos_archivos[0]
                actualizar_columnas({'new': nuevos_archivos[0]})
                label_estado.value = f"<b style='color:#0D9488;'>✓ {len(DATOS)} archivo(s) cargado(s)</b>"
            else:
                label_estado.value = "<b style='color:#EA580C;'>⚠️  No se encontraron archivos CSV</b>"
        except Exception as e:
            label_estado.value = f"<b style='color:#DC2626;'>✗ Error al recargar: {str(e)}</b>"


# ── Función para actualizar opciones de columnas ────────────────────────────
def actualizar_columnas(change):
    """Actualiza las opciones de X, Y y Color cuando cambia el archivo."""
    archivo = change['new']
    if archivo in DATOS:
        df = DATOS[archivo]
        columnas = ['—'] + df.columns.tolist()
        
        selector_x.options = columnas
        selector_y.options = columnas
        selector_color.options = columnas
        
        # Seleccionar primeros valores por defecto
        if len(columnas) > 1:
            selector_x.value = columnas[1]
            selector_y.value = columnas[2] if len(columnas) > 2 else columnas[1]
            selector_color.value = '—'
        
        label_estado.value = f"<b style='color:#0D9488;'>✓ Archivo '{archivo}' cargado ({df.shape[0]} filas × {df.shape[1]} cols)</b>"


# ── Función para generar gráfico ───────────────────────────────────────────
def generar_grafico(button):
    """Genera y muestra el gráfico solicitado."""
    with output_grafico:
        output_grafico.clear_output(wait=True)
        
        archivo = selector_archivo.value
        eje_x = selector_x.value
        eje_y = selector_y.value
        tipo = selector_tipo.value
        color = selector_color.value
        
        # Validar selecciones
        if archivo not in DATOS or eje_x == '—' or (eje_y == '—' and tipo != 'histograma'):
            label_estado.value = "<b style='color:#DC2626;'>✗ Seleccione archivo, eje X y eje Y válidos</b>"
            print("⚠️  Parámetros inválidos")
            return
        
        try:
            df = DATOS[archivo]
            label_estado.value = "<b style='color:#EA580C;'>⏳ Generando gráfico...</b>"
            
            # Generar gráfico
            widget_img = grafico_automatico(df, eje_x, eje_y, tipo, 
                                           color if color != '—' else None)
            display(widget_img)
            
            label_estado.value = f"<b style='color:#0D9488;'>✓ Gráfico generado: {tipo.upper()}</b>"
        except Exception as e:
            label_estado.value = f"<b style='color:#DC2626;'>✗ Error: {str(e)}</b>"


# ── Función para mostrar estadísticas ───────────────────────────────────────
def mostrar_estadisticas(button):
    """Muestra el resumen estadístico del archivo seleccionado."""
    with output_estadisticas:
        output_estadisticas.clear_output(wait=True)
        
        archivo = selector_archivo.value
        
        if archivo not in DATOS:
            print("⚠️  Seleccione un archivo válido")
            return
        
        try:
            df = DATOS[archivo]
            label_estado.value = "<b style='color:#EA580C;'>⏳ Calculando estadísticas...</b>"
            
            html_resumen = resumen_estadistico_html(df)
            from IPython.display import HTML
            display(HTML(html_resumen))
            
            label_estado.value = f"<b style='color:#0D9488;'>✓ Estadísticas de '{archivo}'</b>"
        except Exception as e:
            label_estado.value = f"<b style='color:#DC2626;'>✗ Error: {str(e)}</b>"


# ── Conectar eventos ────────────────────────────────────────────────────────
selector_archivo.observe(actualizar_columnas, names='value')
btn_generar.on_click(generar_grafico)
btn_estadisticas.on_click(mostrar_estadisticas)
btn_recargar.on_click(recargar_archivos)


# ╔══════════════════════════════════════════════════════════╗
# ║  INTERFAZ VISUAL DEL DASHBOARD                          ║
# ╚══════════════════════════════════════════════════════════╝

# Sección: CONTROLES
seccion_controles = widgets.VBox([
    widgets.HTML("<h3 style='color:#2563EB; border-bottom:2px solid #2563EB; padding-bottom:8px;'>⚙️ CONTROLES</h3>"),
    widgets.HBox([selector_archivo], layout=widgets.Layout(margin='10px 0px')),
    widgets.HBox([selector_x], layout=widgets.Layout(margin='10px 0px')),
    widgets.HBox([selector_y], layout=widgets.Layout(margin='10px 0px')),
    widgets.HBox([selector_tipo], layout=widgets.Layout(margin='10px 0px')),
    widgets.HBox([selector_color], layout=widgets.Layout(margin='10px 0px')),
    widgets.HBox([btn_generar, btn_estadisticas], layout=widgets.Layout(gap='10px', margin='15px 0px')),
    widgets.HBox([btn_recargar], layout=widgets.Layout(margin='10px 0px')),
], layout=widgets.Layout(
    border='2px solid #E5E7EB',
    padding='15px',
    margin='10px 0px',
    width='320px'
))

# Sección: GRÁFICO PRINCIPAL
seccion_grafico = widgets.VBox([
    widgets.HTML("<h3 style='color:#2563EB; border-bottom:2px solid #2563EB; padding-bottom:8px;'>📊 GRÁFICO</h3>"),
    output_grafico
], layout=widgets.Layout(
    border='2px solid #E5E7EB',
    padding='15px',
    margin='10px 0px',
    flex='1'
))

# Sección: ESTADÍSTICAS
seccion_estadisticas = widgets.VBox([
    widgets.HTML("<h3 style='color:#2563EB; border-bottom:2px solid #2563EB; padding-bottom:8px;'>📈 RESUMEN ESTADÍSTICO</h3>"),
    output_estadisticas
], layout=widgets.Layout(
    border='2px solid #E5E7EB',
    padding='15px',
    margin='10px 0px',
    flex='1'
))

# Sección: ESTADO
seccion_estado = widgets.VBox([
    label_estado
], layout=widgets.Layout(
    border='1px solid #D1D5DB',
    padding='12px',
    margin='10px 0px',
    background_color='#F9FAFB'
))

# ╔══════════════════════════════════════════════════════════╗
# ║  LAYOUT PRINCIPAL                                       ║
# ╚══════════════════════════════════════════════════════════╝

# Título principal
titulo = widgets.HTML("""
<div style='text-align:center; padding:20px 0px; background:linear-gradient(135deg, #2563EB, #0D9488); 
            color:white; border-radius:8px; margin-bottom:20px;'>
    <h1 style='margin:0; font-size:28px;'>📊 DASHBOARD INTERACTIVO</h1>
    <p style='margin:5px 0 0 0; font-size:14px; opacity:0.9;'>
        Análisis y Visualización de Datos — Acceso y Rendimiento Universitario en Chile
    </p>
</div>
""")

# Layout principal: controles a la izquierda, gráficos a la derecha
layout_principal = widgets.HBox([
    seccion_controles,
    widgets.VBox([seccion_grafico, seccion_estadisticas])
], layout=widgets.Layout(
    gap='15px'
))

# Contenedor final
dashboard_final = widgets.VBox([
    titulo,
    layout_principal,
    seccion_estado
], layout=widgets.Layout(
    width='100%',
    padding='20px'
))

# ── Mostrar el dashboard ────────────────────────────────────────────────────
display(dashboard_final)

# Inicializar con el primer archivo
if DATOS:
    primer_archivo = list(DATOS.keys())[0]
    selector_archivo.value = primer_archivo
    actualizar_columnas({'new': primer_archivo})

print("Dashboard interactivo cargado ✓  —  Interfaz lista para usar")


Dashboard interactivo cargado ✓  —  Interfaz lista para usar
